In [1]:
import sys
import os
from pathlib import Path
__file__ = os.getcwd()
ROOT_DIR = Path(__file__).parent.parent.parent
print(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR))

/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT


In [2]:
from videodeepsearch.tools.implementation.llm.tool import enhance_textual_query, enhance_visual_query
from videodeepsearch.tools.implementation.persist.orc_tool import update_video_context_orc_agent, orc_synthesize_final_findings
from videodeepsearch.tools.implementation.persist.worker_tool import worker_persist_evidence
from videodeepsearch.tools.implementation.scan.tool import get_segments, get_image, extract_frames_by_time_window, extract_frame_time
from videodeepsearch.tools.implementation.search.tool import get_images_from_caption_query, get_images_from_multimodal_query, get_images_from_visual_query, get_segments_from_event_query
from videodeepsearch.tools.implementation.util.tool import get_related_asr_from_image, get_related_asr_from_video_id

from videodeepsearch.tools.implementation.view.orc_tool import (
    orc_view_work_histories, 
    orc_view_current_evidences_by_worker,
    orc_view_past_evidences_by_worker,
    orc_view_all_agent_in_the_past,
    orc_view_video_context,
    orc_view_get_all_video_ids_context
)

from videodeepsearch.tools.implementation.view.worker_tool import (
    worker_view_all_data_handle, 
    worker_view_results,
    worker_view_my_evidence,
    worker_view_statistics,

)


from videodeepsearch.tools.base.registry import tool_registry, get_registry_tools

In [3]:
tool_names = tool_registry.list_all_tool_name()
print(tool_names)

['enhance_visual_query', 'enhance_textual_query', 'update_video_context_orc_agent', 'orc_synthesize_final_findings', 'worker_persist_evidence', 'get_segments', 'get_images_from_visual_query', 'get_images_from_caption_query', 'get_segments_from_event_query', 'get_images_from_multimodal_query', 'get_related_asr_from_video_id', 'get_related_asr_from_image', 'orc_view_work_histories', 'orc_view_current_evidences_by_worker', 'orc_view_past_evidences_by_worker', 'orc_view_all_agent_in_the_past', 'orc_view_video_context', 'orc_view_get_all_video_ids_context', 'worker_view_all_data_handle', 'worker_view_results', 'worker_view_my_evidence', 'worker_view_statistics']


In [4]:
print(tool_registry.get_all_available_group_tools())

Available group doc 
system: **Group**: UTILITY
        **Description**: Utility tools for query enhancement, video navigation, and frame extraction. Includes tools to expand queries for better retrieval, navigate through video timelines (hop between segments/frames), and extract raw frame images.
        **Usage**: 
        - Query enhancement: Use before search to improve recall (enhance_visual_query, enhance_textual_query)
        - Video navigation: Use after finding matches to explore surrounding context (get_segments, get_image)
        - Frame extraction: Use when you need to visually inspect specific time windows
        **Typical patterns**:
        - Enhance query → search → inspect → navigate adjacent content
        - Find segment → hop backward/forward → verify continuity
        - Extract frames → view directly for detailed analysis
        **Tools**:
        
            - **enhance_visual_query**
            

            - **enhance_textual_query**
            

      

In [5]:
print(tool_registry.get_all_available_bundle_spec())

Availabel bundle spec

            ## Bundle name: Video Evidence Worker

            **Description:**  
            Worker strategy: find → verify → persist visual/textual evidence via iterative search, navigation, and transcript grounding.

            **Roles Included:**  
              - **QUERY_ANALYZER**
  - **SEMANTIC_SEARCHER**
  - **TRANSCRIPT_ANALYZER**
  - **WORKER_RESULT_INSPECTOR**
  - **VIDEO_NAVIGATOR**
  - **WORKER_EVIDENCE_MANAGER**

            

            ## Bundle name: Video Evidence Orchestrator

            **Description:**  
            Top-level orchestrator that plans, delegates, monitors, synthesizes, and delivers final reports across evidence workers.

            **Roles Included:**  
              - **ORCHESTRATOR_PLANNER**
  - **ORCHESTRATOR_WORK_INSPECTOR**
  - **ORCHESTRATOR_EVIDENCE_MANAGER**

            


In [9]:
"""
['enhance_visual_query', 'enhance_textual_query', 'update_video_context_orc_agent', 'orc_synthesize_final_findings', 'worker_persist_evidence', 'get_segments', 'get_images_from_visual_query', 'get_images_from_caption_query', 'get_segments_from_event_query', 'get_images_from_multimodal_query', 'get_related_asr_from_video_id', 'get_related_asr_from_image', 'orc_view_work_histories', 'orc_view_current_evidences_by_worker', 'orc_view_past_evidences_by_worker', 'orc_view_all_agent_in_the_past', 'orc_view_video_context', 'orc_view_get_all_video_ids_context', 'worker_view_all_data_handle', 'worker_view_results', 'worker_view_my_evidence', 'worker_view_statistics']

"""

print(
    tool_registry.get_specific_tool_documentation('get_images_from_visual_query')
)

        Tool Documentation

        **Tool Name:**  
        get_images_from_visual_query

        **Group:**  
        SEARCH_GROUP

        **Bundle Spec:**  
        Video Evidence Worker

        **Description:**  
        get_images_from_visual_query(visual_query: Annotated[str, FieldInfo(annotation=NoneType, required=True, description='A visually descriptive natural-language query. Avoid non-visual elements such as names, numbers, or abstract concepts. Must be in English.')], top_k: Annotated[int, FieldInfo(annotation=NoneType, required=False, default=50, description='Number of top-matching images to retrieve.', metadata=[Ge(ge=1), Le(le=100)])], list_video_id: list[str], user_id: str) -> videodeepsearch.tools.base.middleware.data_handle.DataHandle
This is the middlware tool wrapper! Please notice the input/output middleware doc!
This is the original tool docstring:
Search for images using visual similarity based on a natural-language description. Encodes the input query into a C

In [6]:
worker_partial_params = {
    'user_id': 'user_id',
    'list_video_id': ['list_video_ids'],
}
worker_tools = tool_registry.get_concrete_agent_tools(
    agent_name='WORKER_AGENT', 
    **worker_partial_params
)

orc_partial_params = {'session_id': '123'}
orc_tools = tool_registry.get_concrete_agent_tools(
    agent_name='ORCHESTRATOR_AGENT', 
    **orc_partial_params
)

In [7]:

print(tool_registry.get_specific_tool_documentation(
    tool_name='enhance_visual_query'
), sep='\n\n\n')

[ExtentedToolMetadata(whom_agents=['WORKER_AGENT'], func=<function enhance_visual_query at 0x7095e06339c0>, tool_metadata=ToolMetadata(description='enhance_visual_query(raw_query: Annotated[str, "The user\'s original visual query. This may describe objects, actions, scenes, or any visual intent the user wants to search for in the video. The query must be in English"], variants: Annotated[list[str], \'EXTREME-DETAIL perspective variations to generate. Each variant should be a highly-specific viewpoint/visual-demand description i.e. precise camera perspective, framing, scale, motion, and occlusion details. Each variant should represent an extremely detailed reinterpretation of the original content, emphasizing meaningful visual divergence. Variants may include changes in spatial perspective (e.g., different viewing angles, camera positions, scale shifts), alterations in scene composition, modifications of lighting or atmosphere, and nuanced adjustments to texture, depth, geometry, or str

In [18]:
print(
    tool_registry.list_all_tool_name()
)

['enhance_visual_query', 'enhance_textual_query', 'update_video_context_orc_agent', 'orc_synthesize_final_findings', 'worker_persist_evidence', 'get_segments', 'get_images_from_visual_query', 'get_images_from_caption_query', 'get_segments_from_event_query', 'get_images_from_multimodal_query', 'get_related_asr_from_video_id', 'get_related_asr_from_image', 'orc_view_work_histories', 'orc_view_current_evidences_by_worker', 'orc_view_past_evidences_by_worker', 'orc_view_all_agent_in_the_past', 'orc_view_video_context', 'orc_view_get_all_video_ids_context', 'worker_view_all_data_handle', 'worker_view_results', 'worker_view_my_evidence', 'worker_view_statistics']


In [19]:
print(
    tool_registry.get_specific_tool_documentation('enhance_textual_query')
)

[ExtentedToolMetadata(whom_agents=['WORKER_AGENT'], func=<function enhance_visual_query at 0x7095e06339c0>, tool_metadata=ToolMetadata(description='enhance_visual_query(raw_query: Annotated[str, "The user\'s original visual query. This may describe objects, actions, scenes, or any visual intent the user wants to search for in the video. The query must be in English"], variants: Annotated[list[str], \'EXTREME-DETAIL perspective variations to generate. Each variant should be a highly-specific viewpoint/visual-demand description i.e. precise camera perspective, framing, scale, motion, and occlusion details. Each variant should represent an extremely detailed reinterpretation of the original content, emphasizing meaningful visual divergence. Variants may include changes in spatial perspective (e.g., different viewing angles, camera positions, scale shifts), alterations in scene composition, modifications of lighting or atmosphere, and nuanced adjustments to texture, depth, geometry, or str

In [8]:
print(
    tool_registry.get_all_available_bundle_spec()
)

Availabel bundle spec

            ## Bundle name: Video Evidence Worker

            **Description:**  
            Worker strategy: find → verify → persist visual/textual evidence via iterative search, navigation, and transcript grounding.

            **Roles Included:**  
              - **QUERY_ANALYZER**
  - **SEMANTIC_SEARCHER**
  - **TRANSCRIPT_ANALYZER**
  - **WORKER_RESULT_INSPECTOR**
  - **VIDEO_NAVIGATOR**
  - **WORKER_EVIDENCE_MANAGER**

            

            ## Bundle name: Video Evidence Orchestrator

            **Description:**  
            Top-level orchestrator that plans, delegates, monitors, synthesizes, and delivers final reports across evidence workers.

            **Roles Included:**  
              - **ORCHESTRATOR_PLANNER**
  - **ORCHESTRATOR_WORK_INSPECTOR**
  - **ORCHESTRATOR_EVIDENCE_MANAGER**

            


In [9]:
print(tool_registry.get_bundle_documentation("Video Evidence Orchestrator"))

## Note: If there are any tools that you do not know, please use the given discovery tools to find out. You must not 'guess' the input/output, the docstring, and the signature of the tool.


# Strategy: Video Evidence Orchestrator

Top-level orchestrator that plans, delegates, monitors, synthesizes, and delivers final reports across evidence workers.


## 1. Available Tools by Role

### 🔹 Role: Orchestrator Inspector
   - **Goal**: Read-only tools exclusive to the orchestrator agent. Allows inspection of per-worker evidence accumulation and access to historical synthesized summaries across the current reasoning session.
   - The main purpose of this type of toolbox: Enable the orchestrator to monitor progress, retrieve any worker's collected evidence, and review past round summaries without asking workers directly.
   - **Use these tools**: orc_view_current_evidences_by_worker, orc_view_histories

   - The group dependencies that should be execure before this group: No dependencies, us

In [10]:
from videodeepsearch.agent.worker.custom import CustomFunctionAgent
from videodeepsearch.agent.worker.planner.agent import PLANNER_PROMPT, AGENT_PLANNER_DESCRIPTION
from llama_index.llms.google_genai import GoogleGenAI
from google.genai import types
from dotenv import load_dotenv
load_dotenv(dotenv_path='/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/videodeepsearch/test/.env')

True

In [ ]:
import json
from datetime import datetime
from typing import Any

from rich.console import Console, Group
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich import box
from rich.syntax import Syntax
from rich.rule import Rule
from rich.padding import Padding
from rich.style import Style
from rich.theme import Theme

# Llama Index Imports
from llama_index.core.workflow import Event, StopEvent
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentStream,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStreamStructuredOutput
)

# 🎨 BRIGHT NEON THEME FOR DARK TERMINALS
custom_theme = Theme({
    "timestamp": "grey70",
    "agent.name": "bold bright_cyan",
    "agent.label": "bold white on deep_sky_blue1", 
    "tool.name": "bold hot_pink",
    "tool.label": "bold white on deep_pink2",
    "success": "bold spring_green1",
    "warning": "bold gold1",
    "error": "bold red1",
    "header": "bold white",
})

class EventHandler:
    def __init__(self, console: Console = None):
        self.console = console or Console(theme=custom_theme)
        self.last_agent = None
        self.stream_active = False

    def _get_time(self) -> str:
        return datetime.now().strftime("%H:%M:%S")

    def _handle_agent_input(self, data: AgentInput) -> None:
        agent_name = data.current_agent_name
        input_msgs = data.input
        self.last_agent = agent_name

        msg_preview = Text()
        for msg in input_msgs:
            role_style = "bold yellow" if msg.role == "user" else "bold magenta"
            msg_preview.append(f"{msg.role.upper()}: ", style=role_style)
            msg_preview.append(f"{str(msg.content)[:200]}", style="bright_white")
            if len(str(msg.content)) > 200:
                msg_preview.append("...", style="grey50")
            msg_preview.append("\n")

        # Bright Blue Border
        self.console.print(Padding(Panel(
            msg_preview,
            title=f"[agent.label] 🤖 {agent_name} [/agent.label] [grey70]received input[/grey70]",
            border_style="bold dodger_blue1",
            box=box.ROUNDED,
            padding=(0, 2)
        ), (1, 0)))

    def _handle_agent_stream(self, data: AgentStream) -> None:
        if not self.stream_active:
            self.console.print()
            self.console.print(f"[agent.name]⚡ {data.current_agent_name}[/agent.name] [grey70]is working...[/grey70]")
            self.stream_active = True

        delta = data.delta
        thinking_delta = data.thinking_delta

        # Thinking: Lighter purple/grey (readable but distinct)
        if thinking_delta:
            self.console.print(thinking_delta, end='', style="italic plum2")
        # Output: Bright Neon Cyan
        elif delta:
            self.console.print(delta, end='', style="bold cyan1")
        # Heartbeat
        else:
            self.console.print(".", end="", style="dim grey50")

    def _handle_agent_output(self, data: AgentOutput) -> None:
        if self.stream_active:
            self.console.print("\n")
            self.stream_active = False

        agent_name = data.current_agent_name
        response = data.response
        tool_calls = data.tool_calls

        content_group = []
        
        if response.content:
            content_group.append(Text(response.content, style="bright_white"))
        
        if tool_calls:
            t_text = Text("\n🛠️  Triggered Tools:", style="bold gold1")
            for tc in tool_calls:
                t_text.append(f"\n  • {tc.tool_name}", style="khaki1")
            content_group.append(t_text)

        # Bright Green Border
        self.console.print(Padding(Panel(
            Group(*content_group),
            title=f"[agent.label] 💬 {agent_name} [/agent.label] [success]Final Answer[/success]",
            border_style="bold spring_green3",
            box=box.HEAVY_EDGE,
            padding=(1, 2)
        ), (1, 0)))

    def _handle_tool_call(self, data: ToolCall) -> None:
        if self.stream_active:
            self.console.print("\n")
            self.stream_active = False

        tool_name = data.tool_name
        tool_id = data.tool_id
        tool_kwargs = data.tool_kwargs

        table = Table(box=box.SIMPLE, show_header=True, header_style="bold gold1", expand=True)
        table.add_column("Argument", style="bold cyan", ratio=1)
        table.add_column("Value", style="bright_white", ratio=3)

        if tool_kwargs:
            for key, value in tool_kwargs.items():
                table.add_row(key, str(value))
        else:
            table.add_row("-", "[grey50]No parameters[/grey50]")

        # Bright Gold Border
        self.console.print(Padding(Panel(
            table,
            title=f"[tool.label] 🔧 Calling: {tool_name} [/tool.label] [grey70]({tool_id})[/grey70]",
            border_style="bold gold1",
            box=box.ROUNDED,
            padding=(0, 1)
        ), (0, 0, 1, 0)))

    def _handle_tool_call_result(self, data: ToolCallResult):
        tool_name = data.tool_name
        tool_output = data.tool_output
        is_error = tool_output.is_error

        # High Contrast Red vs Bright Green
        style = "bold red1" if is_error else "bold chartreuse3"
        border_style = "red1" if is_error else "chartreuse3"
        icon = "❌" if is_error else "✅"
        
        content_str = str(tool_output)
        try:
            parsed = json.loads(content_str)
            # Monokai is good, but 'ansi_dark' sometimes pops more on black.
            # Keeping monokai but ensuring the surrounding text is bright.
            display_content = Syntax(json.dumps(parsed, indent=2), "json", theme="monokai", word_wrap=True)
        except:
            display_content = Text(content_str, style="bright_white")

        self.console.print(Padding(Panel(
            display_content,
            title=f"[{style}]{icon} Tool Output: {tool_name}[/{style}]",
            border_style=border_style,
            box=box.ROUNDED,
            padding=(1, 2)
        ), (0, 0, 1, 0)))

    def _handle_agent_stream_structured_output(self, data: AgentStreamStructuredOutput) -> None:
        output = data.output
        json_str = json.dumps(output, indent=2)
        syntax = Syntax(json_str, "json", theme="monokai", background_color="default")
        
        # Bright Magenta Border
        self.console.print(Padding(Panel(
            syntax,
            title="[bold magenta1]📊 Structured Output Update[/bold magenta1]",
            border_style="bold magenta1",
            box=box.DOUBLE,
        ), (0, 0)))

    def _handle_stop_event(self, data: StopEvent) -> None:
        if self.stream_active:
            self.console.print("\n")
            self.stream_active = False

        result = data.result
        
        if isinstance(result, (dict, list)):
            content = Syntax(json.dumps(result, indent=2), "json", theme="monokai", line_numbers=True)
        else:
            content = Text(str(result), style="bold bright_white")

        # Neon Green / Success finish
        self.console.print(Padding(Panel(
            content,
            title="[bold white on spring_green3] 🏁 Workflow Completed [/bold white on spring_green3]",
            subtitle=f"[grey70]Finished at {self._get_time()}[/grey70]",
            border_style="bold spring_green3",
            box=box.DOUBLE_EDGE,
            padding=(2, 4)
        ), (2, 0)))

    def handle_event(self, event_data: Event):
        event_type = event_data['event_type']

        handler_map = {
            AgentInput.__name__: self._handle_agent_input,
            AgentOutput.__name__: self._handle_agent_output,
            AgentStream.__name__: self._handle_agent_stream,
            ToolCall.__name__: self._handle_tool_call,
            ToolCallResult.__name__: self._handle_tool_call_result,
            AgentStreamStructuredOutput.__name__: self._handle_agent_stream_structured_output,
            StopEvent.__name__: self._handle_stop_event
        }

        if event_type in handler_map:
            handler_map[event_type](event_data)

In [12]:
thinking_config = types.ThinkingConfig(
    include_thoughts=True,
    thinking_budget=512
) 

generation_config = types.GenerateContentConfig(
    temperature=0.1,
    top_k=1,
    top_p=1,
    thinking_config=thinking_config
)


llm = GoogleGenAI(
    model='gemini-2.5-flash',
    generation_config=generation_config
)

In [13]:
agent = CustomFunctionAgent(
    name="PLANNER_AGENT",
    description=AGENT_PLANNER_DESCRIPTION,
    system_prompt=PLANNER_PROMPT,
    llm=llm,
    tools=get_registry_tools(),
    streaming=True,
    verbose=True    
)

In [14]:
prompt = """
Query: Tôi muốn tìm hình ảnh của một con chó tên là cậu vàng, và nó liên quan tới 1 tựa game. 
Could you craft up a plan to find these?
"""
console = Console()
event_handler = EventHandler(console)

In [15]:
handler = agent.run(user_msg=prompt)
async for ev in handler.stream_events():
    event_handler.handle_event(ev)
final_output = await handler

╭───────────────────────────────────────  🤖 PLANNER_AGENT  received input ───────────────────────────────────────╮
│  SYSTEM:                                                                                                        │
│  <role>                                                                                                         │
│  You are an expert in designing multi-agent systems for worker agents to conduct video searching, based on the  │
│  orchestrator's demand.                                                                                         │
│  </role>                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│  <primary_objective>                                                                                            │
│  Decompose video search queri...                                                                                │
│  USER:                                                                                                          │
│  Query: Tôi muốn tìm hình ảnh của một con chó tên là cậu vàng, và nó liên quan tới 1 tựa game.                  │
│  Could you craft up a plan to find these?                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚡ PLANNER_AGENT is working...

**Designing the Agent Structure**

I'm currently focused on the agent architecture for this image search task. I'm exploring how to divide the labor 
between agents that can handle visual search and agents that can interpret textual information like the dog's name 
and the game context. Right now, I'm thinking about how to get the agents to pass information to each other 
effectively. This will be key to a successful search!

**Refining Agent Toolset**

I've been working on refining the tools the agents will need. After analyzing available tool categories, I'm 
focusing on search, visual analysis, and context retrieval. I'm drilling down into specific tools within these 
groups to understand their parameters and suitability. Now, I'm designing agent blueprints, specifically a visual 
search agent and a textual search agent.

**Initiating Tool Exploration**

I'm now diving into the system's toolset. My immediate priority is understanding the available tool categories via 
`get_all_available_group_tools()`.  I'll then identify the relevant ones for this task: search, visual analysis, 
and context retrieval. Afterward, I'll examine the specific tools within these groups to finalize my agent 
blueprints, focusing on visual and textual search strategies.

...

 retrieved images and their metadata, verifying both visual and semantic relevance.",
    "5. Persist high-confidence evidence using `worker_persist_evidence` if relevant images are found, including 
the image ID, timestamp, and a brief description of

 why it's relevant.",
    "6. Mark task completion using `worker_mark_evidence` once sufficient evidence is gathered or no further 
relevant results are found."
  ]
}

```

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  💬 PLANNER_AGENT  Final Answer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃  The user wants to find images of a dog named "cậu vàng" related to a game. I will design two agent             ┃
┃  configurations to address this query, leveraging different search modalities:                                  ┃
┃                                                                                                                 ┃
┃  **Solution Approach 1: Caption-Based Search Agent**                                                            ┃
┃  This agent will focus on using the Vietnamese textual query to search for images based on their captions. It   ┃
┃  will first enhance the textual query to improve recall and then use the enhanced query for image retrieval.    ┃
┃                                                                                                                 ┃
┃  **Solution Approach 2: Multimodal Search Agent**                                                               ┃
┃  This agent will attempt to combine both visual and textual cues. It will decompose the user's query into a     ┃
┃  visual description (in English) and a semantic caption query (in Vietnamese) and then use a multimodal search  ┃
┃  tool for more precise results.                                                                                 ┃
┃                                                                                                                 ┃
┃  ---                                                                                                            ┃
┃                                                                                                                 ┃
┃  ### Agent Configuration 1: Caption-Based Search Agent                                                          ┃
┃                                                                                                                 ┃
┃  ```json                                                                                                        ┃
┃  {                                                                                                              ┃
┃    "name": "caption_search_agent",                                                                              ┃
┃    "description": "Agent specializing in finding images based on Vietnamese textual captions, enhancing         ┃
┃  queries for better semantic matching.",                                                                        ┃
┃    "task": "Find images of 'cậu vàng' related to a game by leveraging Vietnamese caption search.",              ┃
┃    "plan": [                                                                                                    ┃
┃      "1. Enhance the raw Vietnamese query 'cậu vàng và game' into multiple detailed semantic variants using     ┃
┃  `enhance_textual_query` to improve search recall.",                                                            ┃
┃      "2. For each enhanced query variant, use `get_images_from_caption_query` to search for relevant images.    ┃
┃  The `caption_query` will be the enhanced variant, `top_k_each_request` and `top_k_final` will be set to 10,    ┃
┃  `weights` will be [0.5, 0.5] for a balanced hybrid search, `list_video_id` will be an empty list (searching    ┃
┃  across all available videos), and `user_id` will be 'user_caugvang_game_search'.",                             ┃
┃      "3. Collect all `DataHandle` outputs from `get_images_from_caption_query`.",                               ┃
┃      "4. Use `worker_view_results` for each `DataHandle` to inspect the retrieved images and their metadata.",  ┃
┃      "5. Persist high-confidence evidence using `worker_persist_evidence` if relevant images are found,         ┃
┃  including the image ID, timestamp, and a brief descrip

╔════════════════════════════════════════════  🏁 Workflow Completed  ════════════════════════════════════════════╗
║                                                                                                                 ║
║                                                                                                                 ║
║    The user wants to find images of a dog named "cậu vàng" related to a game. I will design two agent           ║
║    configurations to address this query, leveraging different search modalities:                                ║
║                                                                                                                 ║
║    **Solution Approach 1: Caption-Based Search Agent**                                                          ║
║    This agent will focus on using the Vietnamese textual query to search for images based on their captions.    ║
║    It will first enhance the textual query to improve recall and then use the enhanced query for image          ║
║    retrieval.                                                                                                   ║
║                                                                                                                 ║
║    **Solution Approach 2: Multimodal Search Agent**                                                             ║
║    This agent will attempt to combine both visual and textual cues. It will decompose the user's query into     ║
║    a visual description (in English) and a semantic caption query (in Vietnamese) and then use a multimodal     ║
║    search tool for more precise results.                                                                        ║
║                                                                                                                 ║
║    ---                                                                                                          ║
║                                                                                                                 ║
║    ### Agent Configuration 1: Caption-Based Search Agent                                                        ║
║                                                                                                                 ║
║    ```json                                                                                                      ║
║    {                                                                                                            ║
║      "name": "caption_search_agent",                                                                            ║
║      "description": "Agent specializing in finding images based on Vietnamese textual captions, enhancing       ║
║    queries for better semantic matching.",                                                                      ║
║      "task": "Find images of 'cậu vàng' related to a game by leveraging Vietnamese caption search.",            ║
║      "plan": [                                                                                                  ║
║        "1. Enhance the raw Vietnamese query 'cậu vàng và game' into multiple detailed semantic variants         ║
║    using `enhance_textual_query` to improve search recall.",                                                    ║
║        "2. For each enhanced query variant, use `get_images_from_caption_query` to search for relevant          ║
║    images. The `caption_query` will be the enhanced variant, `top_k_each_request` and `top_k_final` will be     ║
║    set to 10, `weights` will be [0.5, 0.5] for a balanced hybrid search, `list_video_id` will be an empty       ║
║    list (searching across all available videos), and `user_id` will be 'user_caugvang_game_search'.",           ║
║        "3. Collect all `DataHandle` outputs from `get_images_from_caption_query`.",                             ║
║        "4. Use `worker_view_results` for each `DataHand